# TAAC 2026 Demo Dataset EDA

**目标：** 探索 `demo_1000.parquet`（2026.04.10 更新版），重点关注：
1. 新 schema：flat column layout + 新增 sequence feature
2. 各 feature 的基本统计
3. 序列长度分布 → 截断策略
4. NS feature 的缺失值分析
5. 标签分布
6. 针对 UniRECBlock 设计的关键确认项

In [24]:
from datasets import load_dataset
import pandas as pd
import numpy as np

ds = load_dataset("TAAC2026/data_sample_1000")
df = ds['train'].to_pandas()

# 按命名规则分类
seq_cols = [c for c in df.columns if '_seq_' in c]
label_cols = ['label_type', 'label_time']
ns_cols = [c for c in df.columns if c not in seq_cols + label_cols]

# 按domain分组
from collections import defaultdict
domain_seqs = defaultdict(list)
for c in seq_cols:
    domain = c.split('_seq_')[0]
    domain_seqs[domain].append(c)

print(f"总列数: {len(df.columns)}")
print(f"序列列数: {len(seq_cols)}，共 {len(domain_seqs)} 个domain")
for d, cols in domain_seqs.items():
    ids = sorted([int(c.split('_seq_')[1]) for c in cols])
    print(f"  {d}: {len(cols)}条, feature_ids={ids}")
print(f"NS列数: {len(ns_cols)}")

总列数: 120
序列列数: 45，共 4 个domain
  domain_a: 9条, feature_ids=[38, 39, 40, 41, 42, 43, 44, 45, 46]
  domain_b: 14条, feature_ids=[67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 88]
  domain_c: 12条, feature_ids=[27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 47]
  domain_d: 10条, feature_ids=[17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
NS列数: 73


In [25]:
# label_type 转 binary
df['label'] = (df['label_type'] == 2).astype(int)

print("== label_type 分布 ==")
print(df['label_type'].value_counts())
print(f"\n正样本率: {df['label'].mean():.4f} ({df['label'].mean()*100:.1f}%)")

print("\n== label_time ==")
print(f"min : {df['label_time'].min():.0f}")
print(f"max : {df['label_time'].max():.0f}")
print(f"span: {df['label_time'].max() - df['label_time'].min():.0f} 秒 = "
      f"{(df['label_time'].max() - df['label_time'].min())/3600:.1f} 小时")

== label_type 分布 ==
label_type
1    876
2    124
Name: count, dtype: int64

正样本率: 0.1240 (12.4%)

== label_time ==
min : 1772725027
max : 1772725910
span: 883 秒 = 0.2 小时


In [26]:
print("== 各domain序列长度（取第一条seq为代表）==\n")
representative = {d: cols[0] for d, cols in domain_seqs.items()}

for domain, col in representative.items():
    lengths = df[col].apply(lambda x: len(x) if hasattr(x, '__len__') else 0)
    print(f"{domain} (代表: {col})")
    print(f"  mean={lengths.mean():.0f}, median={lengths.median():.0f}, "
          f"max={lengths.max()}, p95={lengths.quantile(0.95):.0f}, p99={lengths.quantile(0.99):.0f}")

# 验证同一domain内长度完全一致
print("\n== domain内各seq长度是否一致 ==")
for domain, cols in domain_seqs.items():
    ref = df[cols[0]].apply(lambda x: len(x) if hasattr(x,'__len__') else 0)
    all_equal = all(
        (df[c].apply(lambda x: len(x) if hasattr(x,'__len__') else 0) == ref).all()
        for c in cols[1:]
    )
    print(f"  {domain}: all_equal={all_equal}")

== 各domain序列长度（取第一条seq为代表）==

domain_a (代表: domain_a_seq_38)
  mean=701, median=578, max=1888, p95=1673, p99=1793
domain_b (代表: domain_b_seq_67)
  mean=571, median=405, max=1952, p95=1563, p99=1803
domain_c (代表: domain_c_seq_27)
  mean=449, median=322, max=3894, p95=1214, p99=2526
domain_d (代表: domain_d_seq_17)
  mean=1100, median=1036, max=3951, p95=2451, p99=2865

== domain内各seq长度是否一致 ==
  domain_a: all_equal=True
  domain_b: all_equal=True
  domain_c: all_equal=True
  domain_d: all_equal=True


In [27]:
col = 'domain_a_seq_38'

# 统计非零率
nonzero_rates = df[col].apply(
    lambda x: np.count_nonzero(x) / len(x) if hasattr(x,'__len__') and len(x)>0 else 0
)
print(f"== domain_a 稀疏性 ==")
print(f"非零率: mean={nonzero_rates.mean():.4f}, median={nonzero_rates.median():.4f}")
print(f"有效行为数(中位数): {(df[col].apply(lambda x: np.count_nonzero(x) if x is not None and hasattr(x,'__len__') else 0)).median():.0f} 个/用户")

# 0值是否集中在末尾（padding）还是散布（真实空槽）
sample = df[col].iloc[0]
nonzero_pos = np.nonzero(sample)[0]
print(f"\n样本: 序列长={len(sample)}, 非零数={len(nonzero_pos)}")
print(f"非零位置前5: {nonzero_pos[:5]}")
print(f"非零位置后5: {nonzero_pos[-5:]}")
print(f"最后非零位置={nonzero_pos[-1]}，序列末尾={len(sample)-1}")
print(f"结论: {'0是末尾padding' if nonzero_pos[-1] < len(sample)*0.9 else '0散布全序列，是真实空槽'}")

== domain_a 稀疏性 ==
非零率: mean=0.0519, median=0.0398
有效行为数(中位数): 22 个/用户

样本: 序列长=682, 非零数=28
非零位置前5: [ 3  8 45 70 87]
非零位置后5: [649 664 679 680 681]
最后非零位置=681，序列末尾=681
结论: 0散布全序列，是真实空槽


In [28]:
# Block 5 替换版
print("== 各domain内各seq的值域 ==\n")
for domain, cols in domain_seqs.items():
    print(f"--- {domain} ---")
    for c in cols:
        all_vals = np.concatenate([x for x in df[c] if x is not None and hasattr(x,'__len__') and len(x)>0])
        nonzero = all_vals[all_vals != 0]
        if len(nonzero) == 0:
            print(f"  {c}: 全为0")
            continue
        print(f"  {c}: min={nonzero.min()}, max={nonzero.max()}, "
              f"unique={len(np.unique(nonzero))}, "
              f"推测={'时间戳' if nonzero.max()>1e9 else 'item_id' if nonzero.max()>1e4 else '类目/属性'}")
    print()

== 各domain内各seq的值域 ==

--- domain_a ---
  domain_a_seq_38: min=-1, max=1201292, unique=17775, 推测=item_id
  domain_a_seq_39: min=1712419199, max=1772725487, unique=321733, 推测=时间戳
  domain_a_seq_40: min=1, max=18, unique=16, 推测=类目/属性
  domain_a_seq_41: min=2, max=11, unique=9, 推测=类目/属性
  domain_a_seq_42: min=1, max=1017, unique=406, 推测=类目/属性
  domain_a_seq_43: min=1, max=3449, unique=1262, 推测=类目/属性
  domain_a_seq_44: min=6, max=15146, unique=3001, 推测=item_id
  domain_a_seq_45: min=3, max=9212, unique=2388, 推测=类目/属性
  domain_a_seq_46: min=1, max=17, unique=11, 推测=类目/属性

--- domain_b ---
  domain_b_seq_67: min=1752645918, max=1772725642, unique=421843, 推测=时间戳
  domain_b_seq_68: min=1, max=27, unique=22, 推测=类目/属性
  domain_b_seq_69: min=115, max=143233599, unique=191930, 推测=item_id
  domain_b_seq_70: min=3, max=733, unique=428, 推测=类目/属性
  domain_b_seq_71: min=3, max=2690, unique=1343, 推测=类目/属性
  domain_b_seq_72: min=3, max=10489, unique=3662, 推测=item_id
  domain_b_seq_73: min=2, max=7238, un

In [29]:
user_int   = [c for c in ns_cols if 'user_int' in c]
user_dense = [c for c in ns_cols if 'user_dense' in c]
item_int   = [c for c in ns_cols if 'item_int' in c]
meta       = ['user_id', 'item_id', 'timestamp']

print(f"NS 字段总数: {len(ns_cols)}")
print(f"  meta:            {len(meta)} 个")
print(f"  user_int_feats:  {len(user_int)} 个")
print(f"  user_dense_feats:{len(user_dense)} 个")
print(f"  item_int_feats:  {len(item_int)} 个")

# 缺失率
print("\n== NS 缺失率（只显示有缺失的字段）==")
for c in ns_cols:
    nr = df[c].isna().mean()
    if nr > 0:
        print(f"  {c}: {nr:.3f}")

# 标量字段统计
scalar_ns = [c for c in user_int + user_dense + item_int
             if df[c].dtype in [np.int64, np.float64, np.int32, np.float32]]
if scalar_ns:
    print("\n== 标量NS字段统计 ==")
    print(df[scalar_ns].describe().round(2))

NS 字段总数: 73
  meta:            3 个
  user_int_feats:  46 个
  user_dense_feats:10 个
  item_int_feats:  14 个

== NS 缺失率（只显示有缺失的字段）==
  user_int_feats_3: 0.030
  user_int_feats_4: 0.030
  user_int_feats_15: 0.139
  user_int_feats_48: 0.002
  user_int_feats_49: 0.007
  user_int_feats_50: 0.004
  user_int_feats_51: 0.001
  user_int_feats_52: 0.001
  user_int_feats_53: 0.001
  user_int_feats_54: 0.368
  user_int_feats_55: 0.019
  user_int_feats_56: 0.019
  user_int_feats_57: 0.031
  user_int_feats_58: 0.150
  user_int_feats_59: 0.150
  user_int_feats_60: 0.592
  user_int_feats_62: 0.070
  user_int_feats_63: 0.070
  user_int_feats_64: 0.070
  user_int_feats_65: 0.080
  user_int_feats_66: 0.086
  user_int_feats_80: 0.200
  user_int_feats_82: 0.204
  user_int_feats_86: 0.692
  user_int_feats_89: 0.055
  user_int_feats_90: 0.091
  user_int_feats_91: 0.450
  user_int_feats_92: 0.494
  user_int_feats_93: 0.171
  user_int_feats_94: 0.521
  user_int_feats_95: 0.318
  user_int_feats_96: 0.678
  user_

In [30]:
print("=" * 60)
print("UniRECBlock 设计参数确认")
print("=" * 60)

print("\n[序列输入]")
for domain, cols in domain_seqs.items():
    rep = cols[0]
    lengths = df[rep].apply(lambda x: len(x) if hasattr(x,'__len__') else 0)
    p95 = lengths.quantile(0.95)
    # domain_a特殊处理：非零率
    if domain == 'domain_a':
        nz = df[rep].apply(lambda x: np.count_nonzero(x) if hasattr(x,'__len__') else 0)
        print(f"  {domain}: {len(cols)}个字段, 原始长p95={p95:.0f}, "
              f"有效行为中位数={nz.median():.0f} → 建议提取非零后截断到64")
    else:
        print(f"  {domain}: {len(cols)}个字段, 长度p95={p95:.0f} → 建议截断到128")

print(f"\n  → concat后总长 L_total = 64 + 128×3 = 448")
print(f"  → segment_emb数量 = 4（每个domain一个）")

print("\n[NS输入]")
print(f"  user_int_feats:   {len(user_int)} 个整数字段")
print(f"  user_dense_feats: {len(user_dense)} 个浮点字段")
print(f"  item_int_feats:   {len(item_int)} 个整数字段（target item）")
print(f"  → NS token数 L_ns = {len(user_int)+len(user_dense)+len(item_int)} 个字段")

print("\n[Label]")
print(f"  label = (label_type == 2).int()")
print(f"  正样本率 = 12.4%，建议关注 pos_weight 或上采样")


UniRECBlock 设计参数确认

[序列输入]
  domain_a: 9个字段, 原始长p95=1673, 有效行为中位数=22 → 建议提取非零后截断到64
  domain_b: 14个字段, 长度p95=1563 → 建议截断到128
  domain_c: 12个字段, 长度p95=1214 → 建议截断到128
  domain_d: 10个字段, 长度p95=2451 → 建议截断到128

  → concat后总长 L_total = 64 + 128×3 = 448
  → segment_emb数量 = 4（每个domain一个）

[NS输入]
  user_int_feats:   46 个整数字段
  user_dense_feats: 10 个浮点字段
  item_int_feats:   14 个整数字段（target item）
  → NS token数 L_ns = 70 个字段

[Label]
  label = (label_type == 2).int()
  正样本率 = 12.4%，建议关注 pos_weight 或上采样


# TAAC 2026 数据集分析：容易踩坑的地方

> 基于 `TAAC2026/data_sample_1000`（2026.04.10更新版）的EDA结论，供队友快速上手。

---

## 一、数据集整体结构

每一行 = **一次广告曝光事件**，包含三类信息：

```
用户是谁        → user_id
曝光了哪个广告   → item_id
有没有转化       → label_type
曝光前的历史行为 → 45条序列字段
```

字段总览：

| 类型 | 字段数 | 说明 |
|---|---|---|
| 序列字段（S） | 45条 | 用户历史行为，按4个domain分组 |
| 用户整数特征 | 46个 | `user_int_feats_*` |
| 用户浮点特征 | 10个 | `user_dense_feats_*` |
| 广告整数特征 | 14个 | `item_int_feats_*` |
| 标签 | 2个 | `label_type`, `label_time` |

---

## 二、序列字段：最容易误解的地方

### ❌ 错误理解

> "有45条独立的行为序列"

### ✅ 正确理解

**45条序列按4个domain分组，同一domain内的所有序列在时间轴上严格对齐，描述的是同一批行为的不同属性。**

```
domain_b 第t步行为：
  domain_b_seq_67[t] = 1772720154  ← 时间戳
  domain_b_seq_69[t] = 88888       ← item_id
  domain_b_seq_68[t] = 5           ← 类目
  ...（共14个属性）

→ 第t步 = 用户的第t次"domain_b类型行为"，14个字段描述这一次行为的完整信息
```

本质上，每个domain是一个 `[T, 字段数]` 的行为矩阵，不是14条独立序列。

### 各domain结构

| domain | 字段数 | 时间戳字段 | item_id字段数 | category字段数 | 序列长度(median) |
|---|---|---|---|---|---|
| domain_a | 9  | seq_39 | 2 | 6 | 578 |
| domain_b | 14 | seq_67 | 6 | 7 | 405 |
| domain_c | 12 | seq_27 | 4 | 7 | 322 |
| domain_d | 10 | seq_26 | 3 | 6 | 1036 |

**注意：字段编号（17、18、27...）是全局编号，不代表字段在domain内的顺序，也不代表语义。**

---

## 三、序列长度：每个用户不一样

```
domain_b: median=405, p95=1563, max=1952
```

这里的数字是**序列的长度**（有多少条历史记录），不是序列元素的值。

- 不同用户的序列长度不同（有的用户历史多，有的少）
- 同一用户在同一domain内，所有字段的长度完全一致（已验证）
- 同一用户跨domain的长度不一样

→ **训练前必须截断+padding，统一到固定长度**

---

## 四、domain_a 特殊：稀疏序列

domain_a 和其他domain不一样：

```
domain_a_seq_38 样本：
[0, 0, 0, 764162, 0, 0, 502217, 0, 0, ...]
```

- **0 不是padding，是真实的"该时间槽无行为"**
- 非零率只有 ~4%，即每100个时间槽只有4次真实行为
- 有效行为数中位数只有约23个，但序列长度中位数是578

→ **建议处理方式：提取 >0 的位置（只保留真实行为），截断到最近64个，同时加positional encoding保留时间位置信息**

---

## 五、特殊值 -1

以下字段存在 `-1` 值：
`domain_a_seq_38`, `domain_b_seq_74`, `domain_b_seq_76`, `domain_b_seq_88`, `domain_c_seq_31`, `domain_c_seq_37`

**-1 是 unknown/缺失 的语义，不是 padding。**

padding 的定义因 domain 而异：
- **domain_b/c/d**：密集序列，截断后长度不足时补的 0 是 padding
- **domain_a**：0 是真实空槽（无行为），不是 padding

→ embedding lookup 前必须对 -1 单独处理，映射到专用的 UNK token index，不能让负数直接进embedding table（会报错或越界）

---

## 六、超大 vocab 问题（待baseline确认）

部分 item_id 字段的 vocab size 极大：

| 字段 | 最大值 | embedding table大小(@dim=64) |
|---|---|---|
| domain_c_seq_47 | 278,677,639 | ~71 GB |
| domain_b_seq_69 | 143,233,599 | ~37 GB |

**直接建embedding table放不进显存，必须用hash trick或frequency truncation。**

具体方案等官方baseline代码确认，**vocab映射必须和baseline保持一致**，否则特征空间不同，没有可比性。

---

## 七、时间戳字段不能直接embedding

每个domain有一条时间戳seq（值在17亿量级，Unix时间戳）。

```
domain_b_seq_67 样本: [1772720154, 1772705768, 1772695970, ...]
label_time: ~1772726000
```

**不能对时间戳做embedding lookup**（vocab太大，且是连续值）。

→ 建议转成相对时间差再使用：
```python
delta_t = label_time - seq_timestamp   # 距曝光时刻过了多久（秒）
delta_t_encoded = log(delta_t + 1)     # log压缩后分桶或直接用
```

---

## 八、label 处理

```python
# label_type ∈ {1, 2}，不是 {0, 1}
# label_type=1 → 未转化（负样本，占87.6%）
# label_type=2 → 转化（正样本，占12.4%）

label = (df['label_type'] == 2).astype(int)
```

**注意正样本率只有12.4%，训练时要考虑类别不平衡（pos_weight或上采样）。**

`label_time` 是曝光时刻的时间戳，不是标签本身，可以用来计算时间差特征。

---

## 九、NS特征注意事项

**user_int_feats（46个）和 item_int_feats（14个）**
- 字段编号不连续（原始特征有些被脱敏删掉了），编号数字本身没有意义
- 每个字段独立做embedding lookup，不能共享embedding table
- item_int_feats 是候选广告的特征，在模型里要单独保留一份当 target token

**user_dense_feats（10个）**
- 真正的连续浮点值，做标准化，不做embedding
- 标准化的mean/std必须在训练集上统计，不能用测试集

---

## 十、数据处理总流程

```
原始数据
  ↓
1. label: (label_type==2).int()
2. NS整数字段: -1/缺失处理 → embedding lookup → 各字段向量
   NS浮点字段: 标准化
   item字段: 单独保留为 target token
  ↓
3. 序列字段（每个domain）:
   a. -1 → UNK index
   b. domain_a: 提取非零位置 → 截断最近64步
      domain_b/c/d: 截断最近128步
   c. 不足的补0 + 生成padding mask
   d. 各字段独立embedding（时间戳字段转相对时间差）
   e. 同domain内所有字段embedding concat → Linear → D维token
      → 每个domain得到一条 [T, D] 的序列
  ↓
4. 4条domain序列 concat + segment_emb → S [B, 448, D]
   NS特征 concat → NS [B, L_ns, D]
   target token → [B, 1, D]
  ↓
5. 进模型
```

---

## 十一、等baseline出来再确定的事项

- [ ] vocab映射方案（hash size 或 frequency truncation 的具体参数）
- [ ] 官方评估指标是全局AUC还是UAUC
- [ ] 平台提交方式
- [ ] 完整训练集的数据规模（demo只有1000样本）

**vocab映射方案必须和baseline一致，其他的可以自己调整。**